In [2]:
import numpy as np
import time

# 1. Define Option Parameters
num_paths = 10_000_000
S0, K, r, sigma, T = 100.0, 105.0, 0.05, 0.2, 1.0

print(f"Starting execution of {num_paths:,} paths on the CPU...")
start_time = time.time()

# 2. Vectorized Generation on the CPU (Uses system RAM instead of GPU VRAM)
# Generates 10 million random variables sequentially/in memory blocks
Z = np.random.standard_normal(num_paths)

# 3. Geometric Brownian Motion Math executed via CPU vector instructions
ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

# 4. Evaluate Financial Payoffs: max(ST - K, 0)
payoffs = np.maximum(ST - K, 0)

# 5. Average and discount back to present value
cpu_price = np.mean(payoffs) * np.exp(-r * T)
cpu_time = time.time() - start_time

print(f"\n=== NATIVE CPU RESULTS (WITH NUMPY) ===")
print(f"Calculated Option Price: ${cpu_price:.4f}")
print(f"Actual CPU Execution Time: {cpu_time:.4f} seconds")

Starting execution of 10,000,000 paths on the CPU...

=== NATIVE CPU RESULTS (WITH NUMPY) ===
Calculated Option Price: $8.0231
Actual CPU Execution Time: 0.5349 seconds


In [3]:
import math
import random
import time

# 1. Define Option Parameters
num_paths = 10_000_000
S0 = 100.0   # Current Stock Price
K = 105.0    # Strike Price
r = 0.05     # Risk-free interest rate
sigma = 0.20 # Market Volatility
T = 1.0      # Time to maturity

print(f"Starting pure sequential CPU execution of {num_paths:,} paths (No NumPy)...")
start_time = time.time()

# Pre-calculate constant components of the Geometric Brownian Motion to optimize the loop
drift = (r - 0.5 * sigma ** 2) * T
volatility_scale = sigma * math.sqrt(T)

total_payoff = 0.0

# 2. Pure Sequential Loop
# This forces the CPU to execute every single path step-by-step
for i in range(num_paths):
    # Generate a single standard normal random variable using the Box-Muller transform natively
    Z = random.gauss(0.0, 1.0)

    # Calculate terminal stock price for this specific path
    ST = S0 * math.exp(drift + volatility_scale * Z)

    # Calculate payoff: max(ST - K, 0)
    payoff = ST - K
    if payoff > 0.0:
        total_payoff += payoff

    # Print progress every 2.5 million paths so you know the script hasn't frozen
    if (i + 1) % 2_500_000 == 0:
        print(f"   Processed {i + 1:,} paths...")

# 3. Average the total accumulated payoffs and discount back to present value
average_payoff = total_payoff / num_paths
pure_cpu_price = average_payoff * math.exp(-r * T)
pure_cpu_time = time.time() - start_time

print(f"\n=== PURE CPU RESULTS (NO NUMPY) ===")
print(f"Calculated Option Price: ${pure_cpu_price:.4f}")
print(f"Actual Pure CPU Execution Time: {pure_cpu_time:.4f} seconds")

Starting pure sequential CPU execution of 10,000,000 paths (No NumPy)...
   Processed 2,500,000 paths...
   Processed 5,000,000 paths...
   Processed 7,500,000 paths...
   Processed 10,000,000 paths...

=== PURE CPU RESULTS (NO NUMPY) ===
Calculated Option Price: $8.0209
Actual Pure CPU Execution Time: 10.1036 seconds
